# PropNavigator - Model Error Analysis

**Diagnostic notebook - not part of the training pipeline.** Re-run it whenever the
model is retrained.

It loads the deployed model and asks: *where is the model accurate, where does it
struggle, and are its errors random or systematic?* Broken down by price bracket,
property type and sector, with a look at the handful of rows that dominate the
squared-error metrics.

The test split is imported from the pipeline module, so the rows are identical to
the ones the reported metrics come from.

## 1. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error, mean_absolute_percentage_error, r2_score,
)

pd.set_option('display.max_rows', 120)
plt.rcParams['figure.figsize'] = (8, 4)

ROOT = Path.cwd()
while not (ROOT / 'artifacts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.model_building.model_building import (
    create_train_val_test_split, inverse_transform_target, MODEL_PATH,
)

df = pd.read_csv(ROOT / 'data/fs/feature_selected_properties.csv')
X_train, X_val, X_test, y_train_log, y_val_log, y_test_log = create_train_val_test_split(df)
print('Test set:', X_test.shape)

## 2. Load the deployed model and predict

In [ ]:
bundle = joblib.load(ROOT / MODEL_PATH)
pipeline = bundle['pipeline']
print('Deployed model:', bundle['model_name'],
      '| saved test MAPE:', bundle['test_mape_percent'], '%',
      '| trained:', bundle['trained_at'])

y_true = inverse_transform_target(y_test_log).values
y_pred = inverse_transform_target(pipeline.predict(X_test))

residuals = y_true - y_pred                    # + means under-predicted
signed_pct = residuals / y_true * 100          # + means under-predicted
abs_pct = np.abs(signed_pct)

test = X_test.copy()
test['y_true_cr'] = y_true
test['y_pred_cr'] = y_pred
test['residual_cr'] = residuals
test['signed_pct'] = signed_pct
test['abs_pct'] = abs_pct

## 3. Overall metrics

The mean and the median tell different stories. MAPE is the mean of absolute
percentage errors; the median shows what a *typical* property experiences.

In [ ]:
summary = pd.DataFrame([
    ('MAPE (%)',           round(abs_pct.mean(), 2)),
    ('Median APE (%)',     round(float(np.median(abs_pct)), 2)),
    ('MAE (Cr)',           round(mean_absolute_error(y_true, y_pred), 3)),
    ('R2',                 round(r2_score(y_true, y_pred), 4)),
    ('P90 error (%)',      round(float(np.percentile(abs_pct, 90)), 1)),
    ('P95 error (%)',      round(float(np.percentile(abs_pct, 95)), 1)),
    ('Max error (%)',      round(float(abs_pct.max()), 0)),
    ('Mean bias (% of mean price)', round(residuals.mean() / y_true.mean() * 100, 2)),
    ('Test count',         int(len(y_true))),
], columns=['metric', 'value'])
summary

Half of all predictions land within roughly 6.5% of the true price. The headline
MAPE is pulled upward by a long right tail - 5% of properties are off by more than
30%. Those two facts together are the honest description of the model.

The mean bias is small but positive: predictions run slightly low on average. That
is the retransformation effect of training in log space and is measured
separately in the model-building notebook.

## 4. Prediction interval coverage

The pipeline calibrates an error band on the **validation** set and stores it in
the artifact. Does that band actually hold on test?

In [ ]:
q = bundle['residual_quantiles']
rel = (y_true - y_pred) / y_pred          # same definition the pipeline uses

coverage = pd.DataFrame([
    ('90% band', f"[{q['q05']:+.3f}, {q['q95']:+.3f}]",
     round(((rel >= q['q05']) & (rel <= q['q95'])).mean() * 100, 1), 90),
    ('80% band', f"[{q['q10']:+.3f}, {q['q90']:+.3f}]",
     round(((rel >= q['q10']) & (rel <= q['q90'])).mean() * 100, 1), 80),
], columns=['band', 'limits', 'test coverage %', 'target %'])
coverage

The validation-calibrated band transfers to test almost exactly. That is the
result you want: the interval the app shows a user is honest.

The limitation is that it is one global band. The next section shows that the
error is far from uniform across price brackets, so a per-band interval would be
tighter for typical properties and wider for the extremes.

## 5. Errors by price bracket - are they random or systematic?

Two things per bracket: how large the errors are (MAPE) and which **direction**
they lean (mean signed error). A model whose errors are random should show bias
near zero everywhere.

In [ ]:
brackets = pd.cut(
    pd.Series(y_true), bins=[0, 1, 3, 5, 10, np.inf],
    labels=['<1 Cr', '1-3 Cr', '3-5 Cr', '5-10 Cr', '10+ Cr'],
)

by_bracket = test.groupby(brackets.values, observed=True).agg(
    count=('y_true_cr', 'size'),
    mape=('abs_pct', 'mean'),
    bias_pct=('signed_pct', 'mean'),
    mae_cr=('residual_cr', lambda r: np.abs(r).mean()),
).round({'mape': 1, 'bias_pct': 1, 'mae_cr': 3})
by_bracket

**The errors are directional.** Negative bias means the model predicts *too high*;
positive means *too low*. Reading down the table: the model over-predicts cheap
properties, is close to unbiased in the 3-5 Cr middle, and under-predicts expensive
ones. That is shrinkage toward the centre of the distribution - a well-known
property of any model fitted by minimising a loss: extreme values are pulled toward
the mean because the model has fewer examples there and regularisation resists
extreme predictions.

It is also the single most actionable finding in this notebook. A bias that
depends on price bracket can be corrected after the fact with a per-bracket
calibration - and the direction tells a user something useful: an estimate above
10 Cr is probably conservative, an estimate below 1 Cr is probably generous.

## 6. Errors by property type

In [ ]:
by_type = test.groupby('property_type').agg(
    count=('y_true_cr', 'size'),
    mape=('abs_pct', 'mean'),
    median_ape=('abs_pct', 'median'),
    bias_pct=('signed_pct', 'mean'),
).round(1).sort_values('mape', ascending=False)
by_type

Independent houses are the weak segment by a wide margin, and the model
over-predicts them substantially. Two reasons, both structural: houses are a small
share of the data, so the model has less to learn from; and the features that
drive house prices - plot size, construction quality, age of the structure - are
either absent or only weakly captured by the columns we have. A super built-up
area figure describes a flat well and a house on a plot poorly.

## 7. Errors by sector - worst and best

In [ ]:
def segment_metrics(labels, min_count=5):
    g = test.groupby(labels, observed=True)
    out = g.agg(
        count=('y_true_cr', 'size'),
        mape=('abs_pct', 'mean'),
        bias_pct=('signed_pct', 'mean'),
        median_price_cr=('y_true_cr', 'median'),
    )
    return out[out['count'] >= min_count].round({'mape': 1, 'bias_pct': 1, 'median_price_cr': 2})


by_sector = segment_metrics(test['sector'].values)

print('Worst 10 sectors by MAPE:')
display(by_sector.sort_values('mape', ascending=False).head(10))
print('\nBest 10 sectors by MAPE:')
display(by_sector.sort_values('mape').head(10))

## 8. The rows that dominate the squared-error metrics

R2 and RMSE are built from squared errors, so a few very large rupee misses can
outweigh thousands of ordinary ones. How concentrated is the error?

In [ ]:
order = np.argsort(-np.abs(residuals))
top20 = order[:20]
rest = np.ones(len(y_true), dtype=bool)
rest[top20] = False

sq = residuals ** 2
print(f'Rows                         : {len(y_true)}')
print(f'Top 20 as share of rows      : {20 / len(y_true) * 100:.2f}%')
print(f'Top 20 as share of squared error : {sq[top20].sum() / sq.sum() * 100:.1f}%')
print()
print(f'R2 with all rows             : {r2_score(y_true, y_pred):.4f}')
print(f'R2 without the top 20        : {r2_score(y_true[rest], y_pred[rest]):.4f}')
print()
print(f'MAPE with all rows           : {abs_pct.mean():.2f}%')
print(f'MAPE without the top 20      : {abs_pct[rest].mean():.2f}%')

A quarter of one percent of the rows carries the majority of the squared error.
Removing them barely moves MAPE but lifts R2 substantially.

This is why the model ranking earlier flipped between MAPE and R2: the two metrics
are effectively measuring different subsets of the data. MAPE describes the typical
property; R2 describes the twenty worst.

In [ ]:
cols = ['property_type', 'sector', 'area', 'bedRoom', 'y_true_cr', 'y_pred_cr', 'residual_cr', 'signed_pct']
worst_rupee = test.iloc[top20][cols].round(2)
worst_rupee

**What the worst rupee errors have in common.** They are almost all ultra-luxury
flats - very large units in sector 42 (Golf Course Road) and Ambience Island,
priced at 60-95 Cr, which the model under-predicts by 20-40 Cr. At that end of
the market there are a few dozen comparable listings in the whole dataset, and
the price depends on things no column captures: which tower, which floor, the
specific developer, the finish.

At least one row is a probable data error rather than a model failure - a builder
floor of ordinary size listed at a price that would put it in the most expensive
building in the city. The model's *disagreement* with the listing is arguably the
right output for a screening tool.

## 9. The worst percentage errors - a different population

Sorting by percentage error instead of rupees surfaces a completely different set
of rows.

In [ ]:
worst_pct = test.sort_values('abs_pct', ascending=False).head(20)[cols].round(2)
worst_pct

These are all **cheap properties predicted far too high** - listings at 20 to 70
lakh that the model prices at 1.3 to 2.4 Cr. Several sit in the `other` sector
bucket, where the location signal has been deliberately coarsened, and at least
one looks like a plot or land listing miscategorised as a house.

Two populations, two failure modes: the rupee tail is the model struggling with
genuine ultra-luxury scarcity; the percentage tail is mostly the model applying
sector-typical pricing to listings that are either mislabelled or genuinely
anomalous. For a screening product, both are listings a buyer should look at
twice - which is exactly the intended use.

## 10. Visual diagnostics

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

# Predicted vs actual, clipped at the 99th percentile so the luxury tail does
# not squash everything else into the corner.
lim = [0, float(np.percentile(y_true, 99))]
ax[0].scatter(y_true, y_pred, s=6, alpha=0.3)
ax[0].plot(lim, lim, 'r--', label='perfect')
ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel('Actual (Cr)'); ax[0].set_ylabel('Predicted (Cr)')
ax[0].set_title('Predicted vs actual (to 99th pct)')
ax[0].legend()

# Signed bias by bracket - the shrinkage picture.
ax[1].bar(by_bracket.index.astype(str), by_bracket['bias_pct'],
          color=['tab:red' if v < 0 else 'tab:blue' for v in by_bracket['bias_pct']])
ax[1].axhline(0, color='k', lw=0.8)
ax[1].set_ylabel('mean signed error %  (+ = under-predict)')
ax[1].set_title('Bias by price bracket')
ax[1].tick_params(axis='x', rotation=30)

# Error distribution, tail clipped for readability.
ax[2].hist(abs_pct[abs_pct < 100], bins=60)
ax[2].axvline(np.median(abs_pct), color='tab:orange', ls='--', label=f'median {np.median(abs_pct):.1f}%')
ax[2].axvline(abs_pct.mean(), color='tab:red', ls='--', label=f'mean {abs_pct.mean():.1f}%')
ax[2].set_xlabel('absolute % error'); ax[2].set_title('Error distribution (<100%)')
ax[2].legend()

plt.tight_layout()
plt.show()

## 11. Save the breakdown

In [ ]:
out = ROOT / 'data/error_analysis'
out.mkdir(parents=True, exist_ok=True)

summary.to_csv(out / 'error_summary.csv', index=False)
coverage.to_csv(out / 'interval_coverage.csv', index=False)
pd.concat([
    by_bracket.reset_index().rename(columns={'index': 'segment'}).assign(segment_type='price_bracket'),
    by_type.reset_index().rename(columns={'property_type': 'segment'}).assign(segment_type='property_type'),
    by_sector.reset_index().rename(columns={'sector': 'segment'}).assign(segment_type='sector'),
], ignore_index=True).to_csv(out / 'segment_metrics.csv', index=False)
worst_rupee.to_csv(out / 'worst_predictions_by_rupee.csv', index=False)
worst_pct.to_csv(out / 'worst_predictions_by_pct.csv', index=False)
print('Saved to', out)

## 12. Findings

**1. A typical prediction is within 6.5%; the headline 10.75% is pulled up by a
tail.** Median absolute error is well below the mean, and 5% of properties are off
by more than 30%. Both numbers belong in any honest description of the model.

**2. The errors are systematic, not random.** The model over-predicts below 1 Cr
by about 13% and under-predicts above 10 Cr by about 7%, with the 3-5 Cr middle
nearly unbiased. That is shrinkage toward the centre, and it is correctable with a
per-bracket calibration. It also gives users a directional hint: high estimates
are probably conservative, low ones probably generous.

**3. Independent houses are the weak segment** - roughly three times the error of
flats, and over-predicted by about 12%. The features available describe flats
well and houses on plots poorly.

**4. Twenty rows carry the majority of the squared error.** They are ultra-luxury
flats on Golf Course Road and Ambience Island, under-predicted by 20-40 Cr, plus
one or two probable data errors. Removing them lifts R2 from 0.93 to 0.96 while
MAPE barely moves - which is exactly why MAPE and R2 disagreed about which model
was best.

**5. The worst percentage errors are a different population**: cheap listings
predicted far too high, many in the coarsened `other` sector bucket, some probably
mislabelled. For a screening tool that flags listings priced unlike their
comparables, these are correct outputs.

**6. The prediction interval is honest.** The validation-calibrated 90% band covers
89.4% of test rows. It is a single global band, though, and finding 2 says a
per-bracket band would serve users better.

### What this argues for next

Per-bracket bias correction is cheap and directly addresses finding 2. Quantile
regression or conformal prediction would give per-property intervals that widen
at the extremes where the model is least certain. And the independent-house
segment would benefit most from a feature the data does not currently have: plot
area separate from built-up area.